# Latency & Overhead Profiling: SSIM Cache-Gating vs ResNet Backward Pass

This notebook benchmarks the computational overhead of **PyTorch-native SSIM gating** versus the cost of **ResNet forward and backward adaptation passes**.

### Core Hypothesis:
- An in-tensor GPU SSIM operation runs in microseconds ($< 0.3$ ms).
- A ResNet backward pass (loss computation, gradient backprop through BN parameters, optimizer update) costs $5 - 15+$ ms.
- Therefore, gating overhead is $\approx 2-5\%$ of backward pass time, making any frame skipped via cache hit an immediate net win in latency and energy efficiency.

In [ ]:
import sys
import time
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Ensure project package is importable
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from gating.similarity import pytorch_ssim, PerceptualHashSimilarity
from models.backbone import build_backbone
from models.tent import Tent

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Benchmarking on Device: {device}")

## 1. Micro-Benchmark: Gating Operators Latency

In [ ]:
# Create sample CIFAR frames (1, 3, 32, 32) in [0, 1]
frame_a = torch.rand(1, 3, 32, 32, device=device)
frame_b = frame_a + torch.randn(1, 3, 32, 32, device=device) * 0.05
frame_b = torch.clamp(frame_b, 0.0, 1.0)

phash = PerceptualHashSimilarity()

# Warmup
for _ in range(20):
    _ = pytorch_ssim(frame_a, frame_b)
    _ = phash(frame_a, frame_b)
if device.type == "cuda":
    torch.cuda.synchronize()

# Benchmark SSIM
num_iters = 200
ssim_times = []
for _ in range(num_iters):
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = pytorch_ssim(frame_a, frame_b)
    if device.type == "cuda": torch.cuda.synchronize()
    ssim_times.append((time.perf_counter() - t0) * 1000.0)

# Benchmark pHash
phash_times = []
for _ in range(num_iters):
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = phash(frame_a, frame_b)
    if device.type == "cuda": torch.cuda.synchronize()
    phash_times.append((time.perf_counter() - t0) * 1000.0)

mean_ssim = np.mean(ssim_times)
mean_phash = np.mean(phash_times)
print(f"Average PyTorch SSIM Gating Time : {mean_ssim:.4f} ms")
print(f"Average Perceptual Hash Time     : {mean_phash:.4f} ms")

## 2. Benchmark: ResNet-18 Forward vs Backward Adaptation Pass

In [ ]:
backbone = build_backbone(name="resnet18", num_classes=10, pretrained=False, dataset="cifar10").to(device)
tent_model = Tent(backbone, lr=1e-3).to(device)

# Warmup
for _ in range(10):
    _ = tent_model.forward(frame_a)
    _ = tent_model.adapt_and_forward(frame_a)
if device.type == "cuda":
    torch.cuda.synchronize()

# Benchmark Forward Pass (Inference Only)
fwd_times = []
for _ in range(num_iters):
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = tent_model.forward(frame_a)
    if device.type == "cuda": torch.cuda.synchronize()
    fwd_times.append((time.perf_counter() - t0) * 1000.0)

# Benchmark Adapt Pass (Forward + Entropy Loss + Backward + Optimizer)
adapt_times = []
for _ in range(num_iters):
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    _, _ = tent_model.adapt_and_forward(frame_a)
    if device.type == "cuda": torch.cuda.synchronize()
    adapt_times.append((time.perf_counter() - t0) * 1000.0)

mean_fwd = np.mean(fwd_times)
mean_adapt = np.mean(adapt_times)
backward_only = max(0.001, mean_adapt - mean_fwd)

print(f"Average Forward Pass Time             : {mean_fwd:.4f} ms")
print(f"Average Full Adaptation Pass Time     : {mean_adapt:.4f} ms")
print(f"Isolated Backward + Optimizer Time    : {backward_only:.4f} ms")
print(f"Gating Overhead / Backward Cost Ratio : {(mean_ssim / backward_only) * 100:.2f} %")

## 3. Analytical Speedup & Breakeven Modeling

Let:
- $T_{\text{fwd}}$ = Forward latency
- $T_{\text{bwd}}$ = Backward + optimizer latency
- $T_{\text{gate}}$ = Cache gating decision latency
- $r$ = Cache Hit Ratio (fraction of frames where adaptation is skipped)

**Baseline TTA Latency (Per Frame):**
$$T_{\text{baseline}} = T_{\text{fwd}} + T_{\text{bwd}}$$

**Cache-Gated TTA Latency (Per Frame):**
$$T_{\text{gated}}(r) = T_{\text{gate}} + T_{\text{fwd}} + (1 - r) T_{\text{bwd}}$$

In [ ]:
hit_ratios = np.linspace(0.0, 1.0, 50)
t_baseline = np.full_like(hit_ratios, mean_adapt)
t_gated = mean_ssim + mean_fwd + (1.0 - hit_ratios) * backward_only
speedup = t_baseline / t_gated

# Breakeven point: T_gated(r_breakeven) = T_baseline => r_breakeven = T_gate / T_bwd
r_breakeven = mean_ssim / backward_only
print(f"Breakeven Hit Ratio: {r_breakeven * 100:.2f}%")
print(f"(If >= {r_breakeven * 100:.2f}% frames are redundant, Cache-Gating strictly accelerates the pipeline!)")

fig, ax1 = plt.subplots(figsize=(8, 5))

color = 'tab:blue'
ax1.set_xlabel('Cache Hit Ratio (Redundant Frames Bypassed)', fontsize=11)
ax1.set_ylabel('Per-Frame Latency (ms)', color=color, fontsize=11)
ax1.plot(hit_ratios, t_baseline, label='Baseline (Every frame adapted)', color='tab:red', linestyle='--', linewidth=2)
ax1.plot(hit_ratios, t_gated, label='Cache-Gated TTA', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.axvline(x=r_breakeven, color='gray', linestyle=':', label=f'Breakeven ({r_breakeven*100:.1f}%)')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Speedup Factor', color=color, fontsize=11)
ax2.plot(hit_ratios, speedup, color=color, linewidth=2, linestyle='-.', label='Speedup')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Theoretical & Empirical Latency Reduction via Cache-Gating', fontsize=12, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center left')
plt.tight_layout()
plt.show()